# C10 - Encoder-Decoder Q&A Chatbot (the finale)

In C9 you fine-tuned an encoder-only model (DistilBERT) and shipped a chatbot that
returns a **label**. That is the whole power of a classifier: it can sort, but it can
never write a sentence back to the user.

Today we cross the last bridge. We load an **encoder-decoder** (sequence-to-sequence)
model, `t5-small`, fine-tune it to answer questions from a context passage, and load it
into a Gradio chatbot that returns a **written answer**. This is the deliverable the whole
course pointed at: a fine-tuned transformer in a simple chatbot.

## Learning objectives

By the end you will be able to:

1. Explain the difference between encoder-only, decoder-only, and encoder-decoder models,
   and say which shape fits which task.
2. Load a pretrained seq2seq model with `AutoModelForSeq2SeqLM` and run it with `.generate()`.
3. Tokenize inputs and targets for seq2seq (the T5 task prefix, label padding with -100).
4. Fine-tune `t5-small` on a SQuAD subset with `Seq2SeqTrainer`.
5. Save the fine-tuned model and load it into a guarded Gradio Q&A chatbot.

## Prerequisites

- C9 (DistilBERT fine-tuning, the HuggingFace `Trainer`, `save_pretrained`, Gradio guard).
- Comfort with `device`, `torch.manual_seed(42)`, and HuggingFace `datasets`.

## Session format

Theory -> Demo -> Lab, four concepts. One guided core lab per concept, plus a labelled
in-notebook stretch and an async homework at the end.

## Runtime

Google Colab. A GPU runtime (Runtime -> Change runtime type -> T4 GPU) makes the fine-tune
finish in a few minutes. The notebook still runs on CPU, just slower.

## Environment setup (next cell)

The next cell installs pinned versions so the notebook behaves the same for everyone. We pin
`numpy<2` because Colab now ships numpy 2.x, and several of our libraries were built against
numpy 1.x. After the install runs, Colab may ask you to **restart the runtime**
(Runtime -> Restart runtime) so the numpy downgrade takes effect. Restart, then run every
cell from the top. We only install what C10 actually uses: `transformers` (the model +
`Seq2SeqTrainer`), `datasets` (SQuAD), `evaluate` (the SQuAD metric, used in the homework),
`accelerate` (the `Trainer` runs on top of it), and `gradio` (the chatbot UI). No gensim or
sentence-transformers here - C10 is pure seq2seq.

In [ ]:
# Install pinned dependencies for Colab.
# numpy<2 is required because Colab ships numpy 2.x and our stack expects numpy 1.x.
# After this runs, if Colab prints a "restart runtime" prompt, do it, then run all cells.
!pip install -q \
    "transformers==4.57.1" \
    "datasets>=2.19,<3" \
    "evaluate" \
    "accelerate" \
    "gradio" \
    "numpy<2"

# Why each package:
# transformers   -> AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, .generate()
# datasets       -> load_dataset("squad")
# evaluate       -> the SQuAD exact-match / F1 metric (used in the homework)
# accelerate     -> the HuggingFace Trainer is powered by accelerate under the hood
# gradio         -> the final Q&A chatbot UI
# numpy<2        -> compatibility pin for the whole course

print("Install step done. If Colab asked to restart the runtime, restart then re-run.")

# Download the small English spaCy pipeline (model weights, separate from pip).
!python -m spacy download en_core_web_sm

# TextBlob/NLTK tokenizer data (punkt_tab) for sentence/word tokenization.
import nltk
nltk.download('punkt_tab')


In [ ]:
# Standard imports for a seq2seq fine-tuning notebook.
import numpy as np
import torch
import transformers
import datasets

from transformers import (
    AutoTokenizer,             # turns text into token ids (and back)
    AutoModelForSeq2SeqLM,     # an encoder-decoder model with a generation head
    DataCollatorForSeq2Seq,    # pads inputs AND labels (labels padded with -100)
    Seq2SeqTrainer,            # the Trainer subclass that knows how to .generate() while evaluating
    Seq2SeqTrainingArguments,  # the training config object
)
from datasets import load_dataset

# Confirm versions so a broken install is caught immediately.
print(f"transformers: {transformers.__version__}")   # expect 4.57.1
print(f"datasets:     {datasets.__version__}")
print(f"numpy:        {np.__version__}")              # expect a 1.x version (<2)

# Device: use the GPU if Colab gave us one, else CPU. Same pattern as C9.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Reproducibility: same seed as the rest of the course.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Environment ready.")

## What are we building today?

Back to the support-platform team. Your C9 DistilBERT chatbot TAGS the sentiment of each incoming message as
POSITIVE or NEGATIVE. Leadership saw it and immediately asked the next question:

> "Great, it sorts the tickets. Can it actually ANSWER the customer from our help docs?"

A classifier cannot answer. It only picks a label. To answer, you need a model that can
**read** a question plus a relevant help-doc passage and **write** a short answer back.

That is a sequence-to-sequence job. We will:

1. Load `t5-small`, a small encoder-decoder model, and watch it answer poorly out of the box.
2. Fine-tune it on SQuAD, a dataset of (question, context, answer) triples, so it learns to
   pull the answer out of the context.
3. Generate answers with `.generate()` and compare before vs after.
4. Save the model and drop it into a Gradio Q&A chatbot.

We will use SQuAD as a stand-in for "your help docs": each example gives a question, a
paragraph of context, and the correct answer span inside that context. That is exactly the
shape of the support-answering task.

In [ ]:
# SQuAD = Stanford Question Answering Dataset. Each example has:
#   question : a natural-language question
#   context  : a paragraph that contains the answer
#   answers  : a dict with 'text' (list of answer strings) and 'answer_start' (list of ints)
# We load the standard "squad" config from the HuggingFace Hub.
raw = load_dataset("squad")
print(raw)  # shows train / validation splits and their sizes

# Look at one example so the data shape is concrete.
example = raw["train"][0]
print("\n--- one SQuAD example ---")
print("QUESTION:", example["question"])
print("CONTEXT :", example["context"][:300], "...")
# The gold answer lives at answers['text'][0]. SQuAD stores it as a list because some
# validation examples have several acceptable answers; for training we take the first.
print("ANSWER  :", example["answers"]["text"][0])

## Concept 1: encoder, decoder, encoder-decoder

Every model in this course has been built from the same transformer block. The difference
between models is which HALF of the architecture they keep.

Think of a human translator working from English to French:

- First she **reads** the whole English sentence and builds an understanding of it in her
  head. That reading-and-understanding stage is the **encoder**. It looks at the entire
  input at once and turns it into a rich set of vectors.
- Then she **writes** the French sentence one word at a time, each new word depending on the
  words she has already written. That writing stage is the **decoder**. It generates output
  token by token (this is called autoregressive generation).

Now the three model shapes:

- **Encoder-only** (BERT, DistilBERT - your C9 model). Keeps only the reading half. Great for
  understanding a whole input and producing a fixed output: a class label, an embedding, a
  span. It cannot write free text.
- **Decoder-only** (the GPT family). Keeps only the writing half. Great at continuing text.
  Out of scope for this course.
- **Encoder-decoder** (T5, BART - today). Keeps both. It READS the input with the encoder and
  WRITES a new sequence with the decoder. This is the right shape when the output is free text
  that depends on the input: translation, summarization, and abstractive question answering.

Rule of thumb:

```python
# Output is a label or a score?         -> encoder-only (classification head)   (C9)
# Output is a written sequence of text? -> encoder-decoder (generation head)     (C10)
```

For our task - read a question + a context, write an answer - we need the encoder-decoder
shape. We will use `t5-small`: small enough to fine-tune in a few minutes, and built around
a clean "text in, text out" idea that makes the encoder-decoder story easy to see.

**Three shapes of a transformer: which half you keep decides the task.**

```mermaid
graph TD
    BLOCK[Same transformer block]
    BLOCK --> ENC[Encoder only<br/>BERT DistilBERT<br/>reads whole input]
    BLOCK --> DEC[Decoder only<br/>GPT family<br/>writes token by token]
    BLOCK --> ED[Encoder decoder<br/>T5 BART<br/>reads then writes]
    ENC --> ENCOUT[Output: label or vector<br/>C9 classifier]
    DEC --> DECOUT[Output: text continuation]
    ED --> EDOUT[Output: new text sequence<br/>C10 QA answer]
```


In [ ]:
# Load the pretrained T5-small tokenizer and seq2seq model.
# AutoModelForSeq2SeqLM gives us a model that already has a generation (LM) head on the
# decoder, so we can call .generate() right away.
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# T5 is genuinely two stacks: an encoder and a decoder. We can see both.
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Total parameters: {n_params:,}")          # ~60M, small by transformer standards
print(f"Has encoder: {model.get_encoder() is not None}")
print(f"Has decoder: {model.get_decoder() is not None}")

# T5's "text in, text out" design means even tasks like translation are phrased as text.
# We tell T5 which task we want by prepending a TASK PREFIX to the input. We will use a
# "question: ... context: ..." prefix for QA. T5 was pretrained with these prefixes, so the
# format is not arbitrary - it matches how the model expects work to be described.
print("\nTokenizer is ready. Prefixes like 'question:' / 'context:' tell T5 what to do.")

### Demo: ask pretrained T5 a question (before fine-tuning)

Before we train anything, let us see how well stock `t5-small` answers a question from a
context. We build the input string with the QA prefix, tokenize it, call `.generate()`, and
decode the output tokens back to text.

`t5-small` was pretrained on a mix of tasks but never specifically taught the SQuAD
question-answering format, so expect a weak, vague, or off-target answer. That weak baseline
is the point: it is the "before" picture that fine-tuning will fix.

In [ ]:
# A helper that turns a (question, context) pair into a T5 input string.
# This is the SAME prefix format we will train on later, so the before/after is a fair test.
def build_input(question, context):
    # The "question: ... context: ..." prefix tells T5 this is a QA task.
    return f"question: {question}  context: {context}"

# Pick one SQuAD validation example to probe.
probe = raw["validation"][0]
q, c = probe["question"], probe["context"]
gold = probe["answers"]["text"][0]

# Tokenize the input string into ids the model can read.
inputs = tokenizer(build_input(q, c), return_tensors="pt", truncation=True, max_length=256).to(device)

# Generate an answer. .generate() runs the decoder autoregressively (token by token)
# until it emits the end-of-sequence token or hits max_new_tokens.
with torch.no_grad():
    out_ids = model.generate(**inputs, max_new_tokens=32)

# Decode ids back to text, skipping the special tokens (pad, eos).
pred = tokenizer.decode(out_ids[0], skip_special_tokens=True)

print("QUESTION    :", q)
print("GOLD ANSWER :", gold)
print("T5 (un-tuned):", pred)   # likely vague or wrong - that is expected before fine-tuning

## Concept 2: tokenizing inputs AND targets

For classification (C9) you tokenized one piece of text and attached an integer label. For
seq2seq there are TWO pieces of text per example:

- the **input**: `"question: {q}  context: {c}"` (what the encoder reads)
- the **target**: the answer string (what the decoder must learn to write)

So we tokenize twice. The input tokens become `input_ids`; the target tokens become
`labels`. Two details matter and are the classic beginner mistakes:

1. **The task prefix.** Always prepend `"question: ... context: ..."`. Drop it and T5 does not
   know what job to do.
2. **Label padding with -100.** Examples in a batch have different lengths, so labels get
   padded. But padding tokens must NOT count toward the loss, or the model wastes capacity
   learning to predict padding. The convention is to set padded label positions to `-100`,
   which PyTorch's cross-entropy ignores. We do NOT do this by hand: `DataCollatorForSeq2Seq`
   sets it for us at batch time.

One more thing you do NOT need to do: you never build `decoder_input_ids` yourself. Give the
model `labels` and T5 automatically shifts them right and prepends its start token to form the
decoder input. This is teacher forcing - during training the decoder is fed the correct
previous answer tokens so it learns the next token at every position.

```python
# The shape of one preprocessed example:
#   input_ids : ids of "question: ... context: ..."
#   labels    : ids of the answer string (padding -> -100 later, done by the collator)
```

**Seq2seq tokenization: tokenize the input and the target, collator pads labels with -100.**

```mermaid
graph TD
    EX[SQuAD example<br/>question context answer]
    EX --> IN["Input string<br/>question: q  context: c"]
    EX --> TGT[Target string<br/>first gold answer]
    IN --> IDS[tokenizer<br/>input_ids]
    TGT --> LAB[tokenizer text_target<br/>labels]
    IDS --> COL[DataCollatorForSeq2Seq]
    LAB --> COL
    COL --> PAD[Pad batch<br/>label padding set to -100<br/>ignored by loss]
```


In [ ]:
# Demo the two-part tokenization on ONE example so the shapes are concrete.
ex = raw["train"][0]
ex_input = build_input(ex["question"], ex["context"])   # the prefixed input string
ex_target = ex["answers"]["text"][0]                     # the answer string

# Tokenize the input (what the encoder reads).
enc = tokenizer(ex_input, max_length=256, truncation=True)

# Tokenize the target (what the decoder must produce). The text_target argument tells the
# tokenizer to treat this string as a generation target rather than as model input.
lab = tokenizer(text_target=ex_target, max_length=32, truncation=True)

print("INPUT string :", ex_input[:90], "...")
print("input_ids len:", len(enc["input_ids"]))
print("TARGET string:", ex_target)
print("label ids    :", lab["input_ids"])
# Decode the label ids back to prove they round-trip to the answer text.
print("labels decode:", tokenizer.decode(lab["input_ids"], skip_special_tokens=True))

### Lab 1: write the seq2seq preprocess function (core, ~12 min)

You will write the function that turns a BATCH of SQuAD examples into the
`input_ids` / `labels` that `Seq2SeqTrainer` consumes. We then `.map(batched=True)` it over
the dataset.

The function receives `examples`, a dict of LISTS (because `batched=True`). For the batch:

1. Build the list of input strings. For each (question, context) pair, produce the prefixed
   string in the same `"question: ... context: ..."` format used in the demo. The helper that
   builds one such string already exists from the demo above; here you build one per pair.
2. Tokenize that list of input strings with truncation and `max_length=max_input` to get the
   model inputs.
3. Build the list of target strings: for each example, the FIRST gold answer (SQuAD stores
   answers as a list; you want the first element of each).
4. Tokenize the targets. Pass them through the tokenizer's TARGET path (the keyword argument
   that marks text as a generation target, shown in the demo cell above) with
   `max_length=max_target`.
5. Attach the tokenized target ids onto the model inputs under the key the Trainer reads for
   supervision (the same key name the demo printed when it showed "label ids").

Return the model-inputs dict. Do not pad here and do not set -100 here - the data collator
does both at batch time.

In [ ]:
# Lengths: inputs can be long (context paragraphs); answers are short.
max_input = 256
max_target = 32

def preprocess(examples):
    # 1. Build one prefixed input string per (question, context) pair in the batch.
    #    Hint: you have a helper that formats a single (question, context) into the prefixed
    #    string; apply it across the parallel lists examples["question"] and examples["context"].
    model_inputs = None  # YOUR CODE  (tokenize the list of prefixed input strings;
                         #             pass truncation=True and max_length=max_input)

    # 2. Build the list of target answer strings (the first gold answer of each example).
    targets = None  # YOUR CODE

    # 3. Tokenize the targets through the tokenizer's TARGET path with max_length=max_target
    #    and truncation=True.
    labels = None  # YOUR CODE

    # 4. Attach the tokenized target ids onto model_inputs under the supervision key the
    #    Trainer reads (the key the demo printed as "label ids").
    None  # YOUR CODE

    return model_inputs

# --- Verification (provided) ---
# We only check the output if the lab actually produced a labelled batch. If preprocess was
# left blank it returns None; rather than crash here, we print a hint and let the silent
# safety-net cell below install a working preprocess so the rest of the notebook runs.
_sample = raw["train"].select(range(4))
_out = preprocess(_sample[:])  # pass the batch dict
if isinstance(_out, dict) and "input_ids" in _out and "labels" in _out:
    assert len(_out["input_ids"]) == 4, "expected 4 examples back from a batch of 4"
    _decoded = tokenizer.decode([t for t in _out["labels"][0] if t >= 0], skip_special_tokens=True)
    print("First decoded target:", _decoded)
    print("Lab 1 verification passed.")
else:
    print("Lab 1 not complete yet - preprocess did not return input_ids + labels.")
    print("The safety-net cell below will install a working preprocess so later cells run.")

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
def preprocess(examples):
    inputs = [build_input(q, c) for q, c in zip(examples["question"], examples["context"])]
    model_inputs = tokenizer(inputs, max_length=max_input, truncation=True)
    targets = [ans["text"][0] for ans in examples["answers"]]
    labels = tokenizer(text_target=targets, max_length=max_target, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
```
</details>


In [ ]:
# Silent safety-net for Lab 1: a runtime guard, NOT the teaching answer. If preprocess does not
# yet produce labels (lab skipped/incomplete), we quietly install a working reference version so
# the .map() in the next cell still runs. The reference implementation is in the collapsed
# "Reveal the safety-net" panel above; this cell only fires when needed.
def _preprocess_works():
    try:
        _probe = preprocess(raw["train"][:2])
        return isinstance(_probe, dict) and "input_ids" in _probe and "labels" in _probe
    except Exception:
        return False

if not _preprocess_works():
    print("Lab 1 incomplete - installing a working reference preprocess so later cells run.")
    def preprocess(examples):
        inputs = [build_input(q, c) for q, c in zip(examples["question"], examples["context"])]
        model_inputs = tokenizer(inputs, max_length=max_input, truncation=True)
        targets = [ans["text"][0] for ans in examples["answers"]]
        labels = tokenizer(text_target=targets, max_length=max_target, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


In [ ]:
# Keep it class-sized so the fine-tune finishes in a few minutes.
# We take a small training slice and a small validation slice. The whole point is to SEE
# fine-tuning move the needle, not to win a leaderboard.
small_train = raw["train"].shuffle(seed=42).select(range(2000))
small_val = raw["validation"].shuffle(seed=42).select(range(200))

# Apply the preprocess function from Lab 1 to both splits.
# remove_columns drops the original text columns so only model tensors remain.
tokenized_train = small_train.map(preprocess, batched=True, remove_columns=small_train.column_names)
tokenized_val = small_val.map(preprocess, batched=True, remove_columns=small_val.column_names)

# The data collator pads each batch and - crucially - sets padded label positions to -100
# so they are ignored by the loss. We never had to write that by hand.
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

print("Tokenized train:", tokenized_train)
print("Tokenized val:  ", tokenized_val)

## Concept 3: fine-tuning with Seq2SeqTrainer

In C9 you used the plain `Trainer` for classification. For generation we use its sibling,
`Seq2SeqTrainer`, which knows how to call `.generate()` when it evaluates (so eval metrics
reflect real generated text, not just next-token loss).

The config object is `Seq2SeqTrainingArguments`. A few choices matter for seq2seq:

- **learning_rate**: T5 likes a clearly higher learning rate than the classification default.
  The classification default (5e-5) trains T5 painfully slowly, so reach for a value a few
  times larger; you will pick the exact number yourself in the lab.
- **predict_with_generate=True**: tells the trainer to actually generate during evaluation.
- **fp16**: half-precision speeds training up a lot on a GPU, but it is only valid on CUDA, so
  we switch it on only when a GPU is present.
- **eval_strategy** (this is the current argument name; the old `evaluation_strategy` is
  deprecated): when to run evaluation. We evaluate once per epoch.

We keep `num_train_epochs` small (1) and the dataset small so the run fits class time.

**The fine-tune loop: Seq2SeqTrainer trains with teacher forcing and generates during eval.**

```mermaid
graph TD
    DATA[Tokenized train set]
    DATA --> ARGS[Seq2SeqTrainingArguments<br/>learning rate  epochs  batch<br/>predict_with_generate]
    ARGS --> TR[Seq2SeqTrainer]
    TR --> FWD[Forward pass<br/>teacher forcing on labels]
    FWD --> LOSS[Cross entropy loss]
    LOSS --> UPD[Backprop and update weights]
    UPD --> TR
    TR --> EVAL[Eval per epoch<br/>generate on val set]
```


In [ ]:
# Demo: build a throwaway Seq2SeqTrainingArguments to see the API shape (NOT the lab answer).
# This is the reference demo for Concept 3, the analogue of the C9 Trainer call. We construct
# a tiny illustrative config with PLACEHOLDER values (deliberately different from what the lab
# asks for) just to show how the pieces wire together: args -> Seq2SeqTrainer. We do NOT train
# here. In Lab 2 you will pick the real values yourself.
_demo_args = Seq2SeqTrainingArguments(
    output_dir="t5-demo-throwaway",      # scratch dir; we never train with this
    eval_strategy="no",                  # demo only, no eval
    save_strategy="no",
    per_device_train_batch_size=2,       # placeholder, NOT the lab value
    learning_rate=1e-3,                  # placeholder, NOT the lab value
    num_train_epochs=1,                  # placeholder
    predict_with_generate=False,         # placeholder; you will reason about this in the lab
    use_cpu=(device.type != "cuda"),     # force CPU off-GPU (avoids MPS crashes on Apple Silicon)
    report_to="none",
)

# A Seq2SeqTrainer is just the model + args + data wired together. Building one does not train;
# .train() (which you call in the lab) is what runs the loop. Here we only confirm the wiring.
_demo_trainer = Seq2SeqTrainer(
    model=model,
    args=_demo_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)
print("Seq2SeqTrainingArguments built. predict_with_generate =", _demo_args.predict_with_generate)
print("Seq2SeqTrainer wired:", type(_demo_trainer).__name__)
print("This is a throwaway demo of the API shape. Lab 2 is where you choose the real values.")


### Lab 2: configure Seq2SeqTrainingArguments and train (core, ~12 min)

Fill in four training-argument values, then build the trainer and call train. The theory cell
above named every value you need; translate each idea into a number or flag here.

1. Set the per-device TRAIN batch size to a small power of two that fits a T4 (drop it if you
   hit out-of-memory).
2. Set the LEARNING RATE to the T5-friendly value from the theory cell, clearly higher than
   the classification default.
3. Set the number of TRAIN EPOCHS so the model makes the smallest number of passes that still
   shows the jump.
4. Turn on generation during evaluation by setting the generate-during-eval flag.

The rest of the arguments (output dir, eval cadence once per epoch, the GPU-only half
precision flag) are provided. Then build the `Seq2SeqTrainer` with the model, the args, the
two tokenized splits, the tokenizer, and the data collator, and call `.train()`.

Expect roughly 3 to 6 minutes on a T4 GPU. When it finishes, the next cells will show the
model answering far better than the un-tuned baseline.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="t5-squad-qa",            # where checkpoints land
    eval_strategy="epoch",               # evaluate once per epoch (current arg name)
    save_strategy="no",                  # skip checkpoint saving to save disk/time in class
    per_device_train_batch_size=None,    # YOUR CODE  (small power of two that fits a T4)
    per_device_eval_batch_size=16,       # provided
    learning_rate=None,                  # YOUR CODE  (the T5-friendly rate from the theory cell)
    num_train_epochs=None,               # YOUR CODE  (fewest passes that still shows the jump)
    predict_with_generate=None,          # YOUR CODE  (the generate-during-eval flag)
    fp16=(device.type == "cuda"),        # half precision only on GPU (provided)
    use_cpu=(device.type != "cuda"),     # force CPU off-GPU (avoids MPS crashes on Apple Silicon)
    logging_steps=50,                    # provided
    report_to="none",                    # no external loggers (provided)
)

# --- Verification + training (provided) ---
# Only verify and train if the four lab values were actually filled in. If they were left as
# None (lab skipped), we skip this block silently and let the safety-net cell below build a
# working config and run a short fine-tune so the rest of the notebook still has a tuned model.
_config_ready = (
    training_args.per_device_train_batch_size is not None
    and training_args.learning_rate is not None
    and training_args.learning_rate > 1e-4
    and training_args.num_train_epochs is not None
    and training_args.predict_with_generate is True
)
if _config_ready:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    print("Config looks good. Starting training...")
    trainer.train()
    print("Training complete.")
else:
    print("Lab 2 not filled in yet - skipping training here; the safety-net cell will handle it.")

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
# The four values to fill in:
per_device_train_batch_size=16   # a small power of two that fits a T4 (drop to 8 on OOM)
learning_rate=3e-4               # T5-friendly, clearly higher than the 5e-5 classification default
num_train_epochs=1               # one pass is enough to show the before/after jump
predict_with_generate=True       # generate during eval, not just next-token loss

# Then build and train:
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized_train, eval_dataset=tokenized_val,
    tokenizer=tokenizer, data_collator=data_collator,
)
trainer.train()
```
</details>


In [ ]:
# Silent safety-net for Lab 2: a runtime guard, NOT the teaching answer. If the training args
# were left blank or invalid (lab skipped), trainer.train() above did not run, so we quietly
# build a working config and run a short fine-tune so the save/reload and Gradio finale
# downstream still have a fine-tuned model to serve. The reference config is in the collapsed
# "Reveal the safety-net" panel above; this cell only fires when needed.
try:
    _needs_train = (
        training_args is None
        or training_args.per_device_train_batch_size is None
        or training_args.learning_rate is None
        or training_args.num_train_epochs is None
        or training_args.predict_with_generate is not True
    )
except NameError:
    # training_args never got bound (e.g. the lab cell errored while building it).
    _needs_train = True
if _needs_train:
    print("Lab 2 incomplete - running a short fallback fine-tune so later cells work.")
    training_args = Seq2SeqTrainingArguments(
        output_dir="t5-squad-qa",
        eval_strategy="epoch",
        save_strategy="no",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=3e-4,
        num_train_epochs=1,
        predict_with_generate=True,
        fp16=(device.type == "cuda"),
        use_cpu=(device.type != "cuda"),
        logging_steps=50,
        report_to="none",
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    trainer.train()
    print("Fallback fine-tune complete.")


## Concept 4: generating answers and seeing the jump

The model is fine-tuned. Now we read answers out of it the same way we did in the "before"
demo: build the prefixed input, tokenize, call `.generate()`, decode. The only difference is
the weights have learned the SQuAD QA pattern.

A couple of `.generate()` knobs worth knowing now (you will experiment with them in the
stretch):

- **max_new_tokens**: a cap on how long the answer can be. SQuAD answers are short, so 32 is
  plenty.
- **num_beams**: 1 is plain greedy decoding (take the single most likely next token every
  step). A value like 4 turns on beam search, which keeps several candidate answers alive and
  usually returns a cleaner result, at a little extra compute.

Reusing the same probe question from the "before" demo, we will see the un-tuned answer next
to the fine-tuned answer. That side-by-side is the whole story of transfer learning in one
print.

**Generate an answer, then compare the un-tuned baseline against the fine-tuned model.**

```mermaid
graph TD
    Q[Question plus context]
    Q --> INP[Build prefixed input<br/>tokenize to tensors]
    INP --> GEN[model.generate<br/>max_new_tokens num_beams]
    GEN --> DEC[Decode ids to text]
    DEC --> BEFORE[Un-tuned t5-small<br/>vague or wrong]
    DEC --> AFTER[Fine-tuned t5-small<br/>matches gold span]
```


### Lab 3: write the answer_question function (core, ~8 min)

Write the single function the chatbot will call. It takes a `question` and a `context` string
and returns the model's answer string. Steps:

1. Build the prefixed input string from the question and context (same helper as before).
2. Tokenize it into tensors on the right device, with truncation and the input max length.
3. Generate output ids. Pass a sensible answer-length cap, and turn on beam search with a few
   beams for cleaner answers.
4. Decode the FIRST generated sequence back to text, skipping special tokens, and return it.

The verification block calls your function on the probe example and prints the fine-tuned
answer next to the un-tuned answer captured earlier, plus the gold answer, so you can see the
jump immediately.

In [ ]:
def answer_question(question, context):
    # 1. Build the prefixed input string.
    text = None  # YOUR CODE

    # 2. Tokenize to tensors on the model's device (truncation=True, max_length=max_input).
    enc = None  # YOUR CODE

    # 3. Generate ids. Use a short answer cap and turn on beam search with a few beams.
    with torch.no_grad():
        out_ids = None  # YOUR CODE

    # 4. Decode the first sequence to text, skipping special tokens, and return it.
    return None  # YOUR CODE

# --- Verification (provided): before vs after ---
# We only show the before/after if answer_question returns a real string. If the lab was left
# blank it returns None; rather than crash here, we print a hint and let the silent safety-net
# cell below install a working answer_question so the finale still runs.
tuned_pred = answer_question(q, c)   # q, c, gold, pred were set in the "before" demo cell
if isinstance(tuned_pred, str) and len(tuned_pred) > 0:
    print("QUESTION      :", q)
    print("GOLD ANSWER   :", gold)
    print("T5 (un-tuned) :", pred)         # weak baseline from before training
    print("T5 (fine-tuned):", tuned_pred)  # should now match or closely match the gold span
    print("\nLab 3 verification passed.")
else:
    print("Lab 3 not complete yet - answer_question did not return a non-empty string.")
    print("The safety-net cell below will install a working answer_question so the finale runs.")

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
def answer_question(question, context):
    text = build_input(question, context)
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input).to(device)
    with torch.no_grad():
        out_ids = model.generate(**enc, max_new_tokens=32, num_beams=4)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)
```
</details>


In [ ]:
# Silent safety-net for Lab 3: a runtime guard, NOT the teaching answer. If answer_question does
# not return a non-empty string (lab skipped/incomplete), we quietly install a working reference
# version so the save sanity-check and the Gradio finale below can call it. The reference
# implementation is in the collapsed "Reveal the safety-net" panel above; this cell only fires
# when needed.
def _answer_question_works():
    try:
        _probe = answer_question(q, c)
        return isinstance(_probe, str) and len(_probe) > 0
    except Exception:
        return False

if not _answer_question_works():
    print("Lab 3 incomplete - installing a working reference answer_question so the finale runs.")
    def answer_question(question, context):
        text = build_input(question, context)
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input).to(device)
        with torch.no_grad():
            out_ids = model.generate(**enc, max_new_tokens=32, num_beams=4)
        return tokenizer.decode(out_ids[0], skip_special_tokens=True)


In [ ]:
# Ship it: save, reload, and serve.
# A chatbot is a separate process from this training notebook, so it must LOAD the model from
# disk. Same pattern as DistilBERT in C9: save BOTH the model and the tokenizer to a folder,
# then reload BOTH. The model folder holds the weights and config; the tokenizer folder holds
# the vocabulary and the rules for turning text into ids. Reload one without the other and the
# model cannot read text.
save_dir = "t5-squad-qa-final"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Saved model + tokenizer to: {save_dir}")

# Reload from disk into fresh objects, exactly as the chatbot process would.
reloaded_model = AutoModelForSeq2SeqLM.from_pretrained(save_dir).to(device)
reloaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)

# Point the live objects at the reloaded ones so answer_question uses the on-disk model.
model = reloaded_model
tokenizer = reloaded_tokenizer

# Sanity-check: the reloaded model answers the probe question.
print("Reloaded model answer:", answer_question(q, c))

**Ship it: save the model, reload it from disk, and serve it in a Gradio chatbot.**

```mermaid
graph TD
    TUNED[Fine-tuned model in memory]
    TUNED --> SAVE[save_pretrained<br/>model and tokenizer to folder]
    SAVE --> DISK[(Saved folder<br/>weights config vocab)]
    DISK --> LOAD[from_pretrained<br/>reload model and tokenizer]
    LOAD --> APP[Gradio Interface]
    APP --> UI[User types question and context]
    UI --> ANS[answer_question writes answer]
```


In [ ]:
# The finale: a Gradio Q&A chatbot. The user types a QUESTION and a CONTEXT passage, and the
# fine-tuned model writes an answer. We guard the import so a no-Gradio environment still runs
# every earlier cell and just prints a fallback instead of launching a UI.
try:
    import gradio as gr

    def chat_fn(question, context):
        # Reuse the exact function from Lab 3, so the UI and the notebook agree.
        if not question or not context:
            return "Please provide both a question and a context passage."
        return answer_question(question, context)

    # Two text inputs (question, context) -> one text output (the generated answer).
    demo = gr.Interface(
        fn=chat_fn,
        inputs=[
            gr.Textbox(label="Question", placeholder="What does the passage say about ...?"),
            gr.Textbox(label="Context", lines=6, placeholder="Paste a help-doc paragraph here"),
        ],
        outputs=gr.Textbox(label="Answer"),
        title="Seq2Seq Q&A Chatbot (fine-tuned t5-small)",
        description="Reads your question + context and writes an answer. The finale of the course.",
    )
    # In Colab, launch creates a public share link automatically.
    demo.launch(share=True)

except ImportError:
    # No Gradio here - fall back to a plain inference call so the cell still works.
    print("Gradio not installed; running a direct inference instead.\n")
    demo_q = "What architecture writes output token by token?"
    demo_c = ("Encoder-decoder models read the input with an encoder and generate the output "
              "with a decoder, one token at a time, which is called autoregressive generation.")
    print("Q:", demo_q)
    print("A:", answer_question(demo_q, demo_c))

## Stretch (fast finishers) and Homework (async)

### Stretch A: decoding-parameter sweep (in notebook, ~10 min)

`.generate()` has knobs that visibly change the answer. On the probe example, generate the
answer several times while varying ONE knob at a time, and read what each does:

- `num_beams=1` (greedy) vs `num_beams=4` (beam search): beam search usually returns a cleaner,
  steadier answer because it keeps several candidates alive instead of committing greedily.
- `max_new_tokens`: a tighter cap forces a shorter answer; too tight and the answer is cut off.
- `no_repeat_ngram_size=2`: forbids any 2-gram from repeating, which kills stutter like
  "the the the".
- `length_penalty`: above 1.0 nudges toward longer answers, below 1.0 toward shorter ones.

```python
# Sketch:
# for nb in [1, 4]:
#     ids = model.generate(**enc, max_new_tokens=32, num_beams=nb)
#     print(nb, tokenizer.decode(ids[0], skip_special_tokens=True))
```

### Stretch B: one prefix, a different task (in notebook, ~5 min)

T5 is "text in, text out". Change ONLY the task prefix from `"question: ..."` to
`"summarize: ..."` and feed it a paragraph (no fine-tuning needed - summarization was in T5's
pretraining). Watch the SAME model do a completely different job. That is the text-to-text
idea made visible.

**Caveat (important):** the `model` in memory is now fine-tuned on SQuAD, which pushed it hard
toward copying short answer spans. That specialization degrades its summarization. So load a
FRESH `t5-small` for this experiment instead of reusing the tuned one:

```python
# Load a pristine t5-small so its pretrained summarization ability is intact.
fresh_tok = AutoTokenizer.from_pretrained("t5-small")
fresh_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)
text = "summarize: " + your_paragraph
enc = fresh_tok(text, return_tensors="pt", truncation=True, max_length=512).to(device)
ids = fresh_model.generate(**enc, max_new_tokens=60, num_beams=4)
print(fresh_tok.decode(ids[0], skip_special_tokens=True))
```

### Homework (async, production-oriented)

1. **Measure it properly.** Use `evaluate.load("squad")` to compute exact-match and F1 on a
   larger held-out slice. Build the predictions list (each item needs the predicted text and
   the example id) and the references list (each needs the gold answers and the id), then call
   `metric.compute(...)`. Report your numbers.
2. **Probe hallucination.** Feed the model a question whose answer is NOT in the supplied
   context (or a context about a different topic). Observe that the model still confidently
   writes an answer. This is the central risk of generative QA: it will invent. Note when it
   does and how badly.
3. **Ground it (RAG).** Recall B5's `semantic_search(query, top_k=3)` over a `corpus`. A real
   assistant does not get the context handed to it - it RETRIEVES the most relevant passage
   first, then answers from that. Sketch a pipeline that uses B5 semantic search to fetch the
   context, then feeds it to `answer_question`. That is a retrieval-augmented generation (RAG)
   QA system, and grounding the answer in retrieved text is the main defense against the
   hallucination you saw in step 2.

**RAG grounding: retrieve the context with semantic search, then answer from it.**

```mermaid
graph TD
    QUERY[User question]
    QUERY --> SEARCH[semantic_search top_k 3<br/>B5 embedder over corpus]
    SEARCH --> CTX[Retrieved context passage]
    QUERY --> QA[answer_question]
    CTX --> QA
    QA --> ANSWER[Grounded written answer]
    ANSWER --> NOTE[Context grounds the model<br/>main defense vs hallucination]
```


## Wrap-up: what you built, and where it leaves you

You fine-tuned an **encoder-decoder** model and shipped it as a Q&A chatbot. That completes
the architecture arc of the whole course:

- **Encoder-only** (B5/B7 embeddings, C9 DistilBERT): read an input, produce a label or a
  vector. One fast forward pass. Use it when the output is a class or a score.
- **Encoder-decoder** (C10 T5): read an input, WRITE a new sequence token by token. Use it
  when the output is free text whose content depends on the input - translation, summarization,
  abstractive question answering.

The trade-off you now understand: generation is autoregressive, so it is slower and costlier
than a single classification pass, and it can hallucinate. The production answer is to ground
it - retrieve a real context (B5 semantic search) and make the model answer from that. Question
+ retrieved context -> generated answer is exactly a RAG assistant.

Two honest notes to carry forward:

- Trained on SQuAD, whose answers are spans copied from the context, T5 mostly learns to COPY
  the right span rather than paraphrase. Its generative muscle really shows when the answer has
  to be synthesized (that is what the `"summarize:"` stretch demonstrates).
- A fine-tuned model is only as good and as fair as its data. The same care about data you
  practiced all course applies here.

### Next: C11 Capstone - ship your own chatbot

In the capstone you choose ONE path - an encoder-only classification chatbot (C9 shape) or an
encoder-decoder Q&A chatbot (C10 shape) - pick a public dataset, fine-tune, evaluate, and ship
it in Gradio. Everything you need is now in your hands.

### Resources

- T5 docs: https://huggingface.co/docs/transformers/model_doc/t5
- Seq2Seq QA example: https://github.com/huggingface/transformers/tree/main/examples/pytorch/question-answering
- Generation strategies: https://huggingface.co/docs/transformers/generation_strategies
- Gradio Interface: https://www.gradio.app/guides/the-interface-class